# Day 3 — Hugging Face & Together AI

---

Yesterday you called two closed-source AI services (OpenAI and Claude). Today: the open-source side.

By the end of this hour you'll:

1. Use **Hugging Face** to run a real AI model on your own laptop — in one line.
2. Understand why laptops aren't practical for the *big* open-source models.
3. Use **Together AI** to call those big models through a simple API — cheap, fast, no GPU needed.

From now on, Together AI is our default provider for teaching.

In [ ]:
!pip install transformers sentence-transformers together python-dotenv --quiet

## 1. What is Hugging Face?

**Hugging Face** is a website where people share open-source AI models — think **"GitHub for AI"**. Over 800,000 models, all free to download.

There's a Python library — `transformers` — that lets you use any of them with 2 lines of code. The magic function is called `pipeline`.

## 2. Your first Hugging Face model — sentiment analysis

This single line downloads a small AI model that tells you whether a sentence is positive or negative.

(First run downloads ~250 MB. After that, it's cached.)

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

for text in ["I love learning about AI!", "This coffee is terrible.", "Meh, it's fine."]:
    result = sentiment(text)[0]
    print(f"{result['label']:8s}  {result['score']:.2%}  ::  {text}")

**What just happened:** you ran a real AI model *locally* on your CPU. No API calls, no cost.

`pipeline(...)` picked a sensible default model, downloaded it, and gave you a Python function you can call.

## 3. Another pipeline — summarization

Same pattern. Just a different task name.

In [ ]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

text = ("The James Webb Space Telescope, launched on December 25, 2021, is the largest optical "
        "telescope in space. It replaces the aging Hubble as NASA's flagship telescope. Its main "
        "mirror is 6.5 meters wide — nearly three times Hubble's — and it observes primarily in "
        "infrared, letting it see through cosmic dust and study the earliest galaxies.")

print(summarizer(text, max_length=40, min_length=10)[0]["summary_text"])

## 4. Embeddings — turning text into numbers

Embeddings turn sentences into lists of numbers so a computer can compare *meaning*, not just spelling. This is what powers search bars in modern apps.

(Preview of Section 5 — you'll build a search engine with this.)

In [ ]:
from sentence_transformers import SentenceTransformer
from numpy import dot
from numpy.linalg import norm

model = SentenceTransformer("all-MiniLM-L6-v2")

vecs = model.encode([
    "I love cats.",
    "Dogs are wonderful.",
    "My favorite programming language is Python.",
])

def cosine(a, b):
    return dot(a, b) / (norm(a) * norm(b))

print("cats vs dogs  :", cosine(vecs[0], vecs[1]))
print("cats vs python:", cosine(vecs[0], vecs[2]))

Cats-and-dogs are closer to each other than either is to Python. The AI figured that out from meaning alone.

## 5. The catch: big models don't fit on a laptop

What you just ran were **small** models (a few hundred MB). Real production models — LLaMA 3, Mistral, Qwen — are 8 to 70 **billion** parameters. On a laptop:

- 8B model → ~5 GB RAM, ~30 seconds per response.
- 70B model → won't fit at all.

Solution: let someone else host them for you. That's **Together AI**.

## 6. Together AI — the middle path

**Together AI** = a service that hosts hundreds of open-source models and gives you one API to call any of them.

Think:
> *"Netflix for AI models"* — instead of downloading them, you stream them.

**Why teams pick it:**
- **Cheap** — often 5–10× cheaper than GPT-4o for similar quality.
- **Model choice** — swap `LLaMA 3.1 8B` for `Mixtral 8x7B` by changing one string.
- **Open source** — you can eventually self-host the same model with no rewrite.

**Get a key:** sign up at [together.ai](https://together.ai) → copy the API key → add to `.env`:

```
TOGETHER_API_KEY=...
```

In [ ]:
import os
from dotenv import load_dotenv
from together import Together

load_dotenv()
assert os.getenv("TOGETHER_API_KEY"), "Add TOGETHER_API_KEY to your .env file first"

client = Together()
MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in 3 languages."}],
)

print(resp.choices[0].message.content)

## 7. Swapping models is one line

| Model string | Size | Good for |
|---|---|---|
| `meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo` | 8B | Cheap default |
| `meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo` | 70B | Higher quality |
| `mistralai/Mistral-7B-Instruct-v0.3` | 7B | Fast, concise |
| `Qwen/Qwen2.5-7B-Instruct-Turbo` | 7B | Great at code |

In [ ]:
for model in [
    "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
    "mistralai/Mistral-7B-Instruct-v0.3",
]:
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Describe the color blue in one sentence."}],
    )
    print(f"[{model.split('/')[-1]}]")
    print(" ", r.choices[0].message.content, "\n")

## Recap

- **Hugging Face** = library of 800k+ open-source AI models; use `pipeline(...)` to run small ones on your laptop.
- Small models are great for classification, summarization, embeddings — CPU is enough.
- Big models (LLaMA 70B, etc.) don't fit on a laptop.
- **Together AI** hosts the big ones and gives you an OpenAI-style API.
- Swap models by changing one string. Best of both worlds: open source + no GPU.

Tomorrow: the art of *asking* the AI the right way — prompt engineering.